# Two-Way Swaps, One Run Command
**Choose a benchmark config. Choose a harness. Run the actual Gym CLI.**

Hermes → Pi on the same SWE-Pro task, then keep Pi and switch to Terminal-Bench 2.1.
One task per pair demonstrates the workflow, not comparative accuracy.

<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

## Setup — outside the demo
Follow README to install Gym and prepare data on a Linux Docker host.
Use this checkout's Python kernel. Setup starts normal Gym services; it does not run tasks.
The visible cells below call `gym` directly, without a custom run alias.

In [ ]:
import atexit
import getpass
import importlib.util
import json
import os
from pathlib import Path
import socket
import subprocess
import sys
import time
from urllib.error import URLError
from urllib.request import urlopen
from uuid import uuid4

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nemo_gym").is_dir())
os.chdir(ROOT)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["RAY_TMPDIR"] = "/tmp"
for name, prompt in [
    ("DEMO_MODEL_URL", "Chat Completions URL (ending /v1): "),
    ("DEMO_MODEL_NAME", "Model name: "),
    ("DEMO_GYM_HOST", "Host address reachable from Docker: "),
]:
    if not os.environ.get(name):
        os.environ[name] = input(prompt).strip()
if not os.environ.get("DEMO_MODEL_KEY"):
    os.environ["DEMO_MODEL_KEY"] = getpass.getpass("Model API key (not saved): ")

spec = importlib.util.spec_from_file_location("swap_setup", ROOT / "examples/harness-swaps/run.py")
helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helpers)
services = None
run_dir = None

def stop_services():
    global services
    if services is None:
        return
    helpers.stop_process(services)
    services = None
    found = subprocess.run(
        ["docker", "ps", "-aq", "--filter", f"label=gym-swap-demo={run_dir.name}"],
        capture_output=True, text=True, check=True,
    ).stdout.split()
    if found:
        subprocess.run(["docker", "rm", "-f", *found], check=True)
    (run_dir / "cleanup.json").write_text(json.dumps({
        "containers_remaining_at_shutdown": found, "demo_containers_removed": True,
    }, indent=2))
    print("Stopped this run's services. Artifacts:", run_dir)
    if found:
        print("Warning: fallback container removal was needed; inspect cleanup.")

def start_services(benchmark):
    global services, run_dir
    stop_services()
    assert benchmark in {"swe-pro", "tb21"}
    assert Path(f"examples/harness-swaps/data/{benchmark}.jsonl").is_file()
    run_dir = ROOT / "results/harness-swaps" / (time.strftime("%Y%m%d-%H%M%S") + "-" + benchmark + "-" + uuid4().hex[:6])
    run_dir.mkdir(mode=0o700, exist_ok=False)
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    os.environ.update(DEMO_HEAD_PORT=str(port), DEMO_RUN_DIR=str(run_dir), DEMO_RUN_ID=run_dir.name)
    command = ["gym", "env", "start", "--config", f"examples/harness-swaps/{benchmark}.yaml"]
    print(" ".join(command))
    try:
        with (run_dir / "services.log").open("w") as log:
            services = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(450):
            if services.poll() is not None:
                raise RuntimeError("Startup failed; inspect services.log")
            try:
                with urlopen(f"http://127.0.0.1:{port}/readyz", timeout=2) as response:
                    if response.status == 200:
                        print("Ready. Run the Gym command below.")
                        return
            except (URLError, TimeoutError):
                pass
            time.sleep(2)
        raise TimeoutError("Startup deadline; inspect services.log")
    except BaseException:
        stop_services()
        raise

def finish_run():
    try:
        summary = helpers.summarize(run_dir, expected=1)
        (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
        print(json.dumps(summary, indent=2))
        assert summary["verification_complete"], "Verification incomplete; inspect saved logs."
    finally:
        stop_services()

atexit.register(stop_services)
print("Setup ready; keys are not printed or saved.")

## 1 · Hermes × SWE-Pro
Start services in the setup cell, then run one real task.

In [ ]:
start_services("swe-pro")

In [ ]:
%%bash
gym eval run --no-serve \
  --config examples/harness-swaps/swe-pro.yaml \
  --agent hermes_agent --limit 1

In [ ]:
finish_run()

## 2 · Swap the harness
Change only `--agent`. The selected task file, model and verifier stay the same.

In [ ]:
start_services("swe-pro")

In [ ]:
%%bash
gym eval run --no-serve \
  --config examples/harness-swaps/swe-pro.yaml \
  --agent pi_agent --limit 1

In [ ]:
finish_run()

## 3 · Swap the benchmark
Keep Pi. Change the benchmark config to `tb21.yaml`.

In [ ]:
start_services("tb21")

In [ ]:
%%bash
gym eval run --no-serve \
  --config examples/harness-swaps/tb21.yaml \
  --agent pi_agent --limit 1

In [ ]:
finish_run()

## What changed?
The three runs use the real `gym eval run` command. No per-pair Python edits.
`--no-serve` collects against services started during setup; changing benchmark configs requires starting the matching services.
Each invocation saves fresh rollouts, health reports and cleanup evidence. A valid zero reward is not an infrastructure failure.

These are preview integrations. One selected task does not certify every pairing or benchmark task.

---
## How it works — separate from the demo
The benchmark YAML binds independent Agent and Resources configs through native EnvironmentServer sessions.
Both harnesses run inside the task sandbox. The flat-row adapter changes data format, not execution to a host CLI.

**Add a harness once:** implement its Agent adapter and runtime/tool mapping.
**Add a benchmark once:** prepare tasks and implement Resources setup, verification and cleanup.
Compatible counterparts then need configuration and real per-pair validation, not another bespoke adapter.

No universal before/after LOC claim: some older pairs were already config-only.
See README and VALIDATION for source pins, actual results and limitations.